# ЛР2 — практика B: несколько таблиц в pandas

**Тема:** ключи, `merge`, `groupby`, масштаб и построение feature table  
**Формат:** guided demo на второй практике темы «Табличные данные»  
**Данные:** маленький пример для механики merge + транзакционный набор на 100 000 строк

> **Центральный вопрос:** что означает одна строка до и после каждой операции?

> **Как работаем с demo.** Не перепечатывайте готовый код. Перед объединением сначала зафиксируйте прогноз: что является одной строкой, какой ключ связывает таблицы и сколько строк ожидается после операции. Затем запускайте ячейку и сравнивайте результат с ожиданием.

## 0. Окружение

Нужен только `pandas`. Первая часть создаёт данные прямо в notebook, вторая читает подготовленные CSV из `../data/`. GPU и интернет не нужны.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("pandas:", pd.__version__)

# 1. Сначала — маленький пример, который можно проверить глазами

Есть три таблицы:

- `customers`: **одна строка = один клиент**;
- `orders`: **одна строка = один заказ**;
- `tickets`: **одна строка = одно обращение в поддержку**.

До запуска предположите, в какой таблице `customer_id` должен быть уникален, а где повторения нормальны.

In [ ]:
customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106],
    "city": ["Kazan", "Moscow", "Kazan", "Perm", "Moscow", "Perm"],
    "segment": ["new", "regular", "regular", "new", "regular", "new"],
})

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "customer_id": [101, 101, 102, 102, 102, 103, 104, 104, 105, 105, 105],
    "amount": [900, 1200, 700, 1600, 1100, 500, 1800, 600, 400, 900, 1300],
})

tickets = pd.DataFrame({
    "ticket_id": [11, 12, 13, 14, 15, 16, 17, 18, 19],
    "customer_id": [101, 102, 102, 103, 104, 104, 105, 105, 106],
    "resolution_min": [15, 40, 25, 10, 60, 35, 20, 45, 30],
})

display(customers)
display(orders.head())
display(tickets.head())

## 1.1. Проверяем ключ, а не предполагаем его уникальность

Уникальность — свойство ключа **в конкретной таблице**. Один и тот же `customer_id` может быть уникальным в справочнике клиентов и многократно повторяться в таблице событий.

In [ ]:
def key_report(df, key):
    return pd.Series({
        "rows": len(df),
        "unique_keys": df[key].nunique(),
        "duplicated_key_rows": int(df[key].duplicated(keep=False).sum()),
    })

report = pd.DataFrame({
    "customers": key_report(customers, "customer_id"),
    "orders": key_report(orders, "customer_id"),
    "tickets": key_report(tickets, "customer_id"),
}).T

display(report)

# 2. `merge`: записываем ожидание о связи прямо в коде

Добавим профиль клиента к каждому заказу. Строка результата всё ещё означает **один заказ**, поэтому число строк не должно измениться.

До запуска предположите, какой `validate` соответствует связи «много заказов → один профиль клиента».

In [ ]:
orders_with_customer = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

display(orders_with_customer.head(8))
print("rows before / after:", len(orders), len(orders_with_customer))
print(orders_with_customer["_merge"].value_counts())

## 2.1. Почему `validate` полезен

Представьте ошибку в справочнике: для `customer_id=102` случайно появилась вторая строка. Обычный merge размножит все заказы этого клиента. `validate="many_to_one"` превращает наше ожидание в автоматическую проверку.

In [ ]:
broken_customers = pd.concat([
    customers,
    customers.loc[customers["customer_id"].eq(102)].assign(segment="duplicate_row"),
], ignore_index=True)

print("duplicate rows:")
display(broken_customers[broken_customers["customer_id"].eq(102)])

try:
    orders.merge(broken_customers, on="customer_id", how="left", validate="many_to_one")
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

# 3. Many-to-many: технически допустимо, но легко изменить смысл строки

И `orders`, и `tickets` содержат много строк на клиента. Прямой merge создаёт все пары «заказ × обращение» внутри каждого `customer_id`.

До запуска выберите клиента с несколькими заказами и несколькими обращениями и предположите число строк результата для него.

In [ ]:
direct = orders.merge(tickets, on="customer_id", how="inner")

counts = pd.DataFrame({
    "orders": orders.groupby("customer_id").size(),
    "tickets": tickets.groupby("customer_id").size(),
    "rows_after_direct_merge": direct.groupby("customer_id").size(),
}).fillna(0).astype(int)
counts["orders_x_tickets"] = counts["orders"] * counts["tickets"]

display(counts)

true_spend = orders.groupby("customer_id")["amount"].sum().rename("true_spend")
wrong_spend = direct.groupby("customer_id")["amount"].sum().rename("after_direct_merge")
display(pd.concat([true_spend, wrong_spend], axis=1))

# 4. Если нужна одна строка на клиента — сначала агрегируем

`groupby + agg` здесь означает не «ещё один метод pandas», а **смену уровня наблюдения**: из множества заказов одного клиента получаем одну строку с признаками.

In [ ]:
order_features = (
    orders.groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_spend=("amount", "sum"),
        avg_order=("amount", "mean"),
    )
)

ticket_features = (
    tickets.groupby("customer_id", as_index=False)
    .agg(
        ticket_count=("ticket_id", "count"),
        avg_resolution_min=("resolution_min", "mean"),
    )
)

features_small = (
    customers
    .merge(order_features, on="customer_id", how="left", validate="one_to_one")
    .merge(ticket_features, on="customer_id", how="left", validate="one_to_one")
)

display(features_small)

# 5. Теперь тот же принцип на 100 000 строк

Маленький пример нужен, чтобы увидеть механику. Но в реальной работе таблицы быстро перестают помещаться на экран. В каталоге данных лежат:

- `clients.csv` — 5000 клиентов;
- `transactions.csv` — 100 000 операций;
- `mcc_codes.csv` — справочник категорий операций;
- `transaction_types.csv` — справочник типов и направления `expense/income`.

Часть клиентов не имеет операций. Кроме того, небольшой набор транзакций содержит MCC-код, которого нет в справочнике — это намеренная проверка качества merge.

In [ ]:
candidate_dirs = [
    Path("../data"),
    Path("data"),
    Path("../../../shared/02_tabular_data/data"),
]
DATA_DIR = next((p for p in candidate_dirs if (p / "transactions.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Не найден каталог данных ЛР2.")

clients = pd.read_csv(DATA_DIR / "clients.csv", parse_dates=["registration_date"])
transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["transaction_time"])
mcc_codes = pd.read_csv(DATA_DIR / "mcc_codes.csv")
transaction_types = pd.read_csv(DATA_DIR / "transaction_types.csv")

for name, df in {
    "clients": clients,
    "transactions": transactions,
    "mcc_codes": mcc_codes,
    "transaction_types": transaction_types,
}.items():
    print(f"{name:18s} shape={df.shape}, memory={df.memory_usage(deep=True).sum()/1024**2:.2f} MiB")

## 5.1. На большом наборе проверки становятся важнее просмотра `head()`

До запуска сформулируйте ожидаемую уникальность каждого ключа:

- `clients.client_id`;
- `transactions.transaction_id`;
- `transactions.client_id`;
- `mcc_codes.mcc_code`;
- `transaction_types.transaction_type_id`.

In [ ]:
checks = pd.Series({
    "clients.client_id unique": clients["client_id"].is_unique,
    "transactions.transaction_id unique": transactions["transaction_id"].is_unique,
    "transactions.client_id unique": transactions["client_id"].is_unique,
    "mcc_codes.mcc_code unique": mcc_codes["mcc_code"].is_unique,
    "transaction_types.id unique": transaction_types["transaction_type_id"].is_unique,
})
print(checks)
print("clients with transactions:", transactions["client_id"].nunique(), "/", len(clients))

# 6. Обогащаем fact-таблицу справочниками

`transactions` — центральная **fact table**: одна строка = одно событие. Справочники добавляют описание, но не должны менять число событий.

Сначала присоединим MCC и явно посмотрим неприсоединившиеся строки.

In [ ]:
tx = transactions.merge(
    mcc_codes,
    on="mcc_code",
    how="left",
    validate="many_to_one",
    indicator="mcc_merge",
)

print("rows before / after:", len(transactions), len(tx))
print(tx["mcc_merge"].value_counts())
display(tx.loc[tx["mcc_merge"].eq("left_only"), ["transaction_id", "mcc_code", "amount"]].head())

tx["category"] = tx["category"].fillna("Unknown")

tx = tx.merge(
    transaction_types,
    on="transaction_type_id",
    how="left",
    validate="many_to_one",
)
tx = tx.merge(
    clients[["client_id", "city", "segment"]],
    on="client_id",
    how="left",
    validate="many_to_one",
)

assert len(tx) == len(transactions)
display(tx.head())

# 7. Время и `pivot_table`: задаём уже содержательные вопросы

После `parse_dates` дата — не строка, а источник признаков. Например, можно посмотреть активность по часу или по дню недели.

А `pivot_table` удобно использовать, когда хотим одновременно разложить одну числовую величину по двум категориям.

In [ ]:
tx["hour"] = tx["transaction_time"].dt.hour
tx["weekday"] = tx["transaction_time"].dt.day_name()

hourly = tx.groupby("hour", as_index=False).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("amount", "sum"),
)
display(hourly.head(8))

city_direction = tx.pivot_table(
    index="city",
    columns="direction",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
display(city_direction.sort_values("expense", ascending=False))

# 8. Строим feature table уровня клиента

Теперь конечная цель снова формулируется как **одна строка = один клиент**. Поэтому сначала агрегируем транзакции, а затем присоединяем результат к полному списку клиентов через `left`.

Обратите внимание: ноль транзакций и неизвестное среднее значение — не всегда одно и то же. Счётчики и суммы для неактивного клиента естественно заполнить нулём, а среднюю сумму операции лучше оставить `NaN`, если операций не было.

In [ ]:
expense = tx.loc[tx["direction"].eq("expense")]
income = tx.loc[tx["direction"].eq("income")]

base_features = (
    tx.groupby("client_id", as_index=False)
    .agg(
        transaction_count=("transaction_id", "count"),
        active_days=("transaction_time", lambda s: s.dt.date.nunique()),
        unique_categories=("category", "nunique"),
    )
)

expense_features = (
    expense.groupby("client_id", as_index=False)
    .agg(
        expense_count=("transaction_id", "count"),
        total_expense=("amount", "sum"),
        avg_expense=("amount", "mean"),
    )
)

income_features = (
    income.groupby("client_id", as_index=False)
    .agg(total_income=("amount", "sum"))
)

features = (
    clients
    .merge(base_features, on="client_id", how="left", validate="one_to_one")
    .merge(expense_features, on="client_id", how="left", validate="one_to_one")
    .merge(income_features, on="client_id", how="left", validate="one_to_one")
)

zero_cols = ["transaction_count", "active_days", "unique_categories", "expense_count", "total_expense", "total_income"]
features[zero_cols] = features[zero_cols].fillna(0)

display(features.head())
print("shape:", features.shape)
print("clients without transactions:", int(features["transaction_count"].eq(0).sum()))

## 8.1. Sanity checks — часть результата

На 100 000 строк уже нельзя проверить корректность глазами. Поэтому фиксируем инварианты, которые должны оставаться истинными независимо от размера данных.

In [ ]:
sanity = pd.Series({
    "one row per client": features["client_id"].is_unique,
    "all clients preserved": len(features) == len(clients),
    "same client ids": set(features["client_id"]) == set(clients["client_id"]),
    "transaction counts preserved": int(features["transaction_count"].sum()) == len(transactions),
    "nonnegative totals": bool((features[["total_expense", "total_income"]] >= 0).all().all()),
})
print(sanity)

# 9. Переход к самостоятельной работе

В самостоятельной части вы работаете с тем же набором, но строите feature table сами и добавляете новые операции:

- диагностику ключей и неприсоединившихся значений;
- временные признаки;
- `groupby + agg`;
- `pivot_table` по категориям расходов;
- `qcut` для сегментации клиентов по общей сумме расходов;
- собственные sanity checks.

Главная проверка остаётся прежней: **после каждой трансформации уметь объяснить, что означает одна строка таблицы.**